In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
population = pd.read_parquet("../Life_tables_data/weekly_population_departements.parquet")
deaths = pd.read_parquet("../Life_tables_data/deaths_weekly.parquet")
weather = pd.read_parquet("../Life_tables_data/weekly_weather.parquet")

In [ ]:
base = "../Life_tables_data/"

nat_pred = pd.read_parquet(base + "national_mortality_predictions.parquet")
brass_pred = pd.read_parquet(base + "brass_mortality_predictions.parquet")
glm_weekly = pd.read_parquet(base + "dep_glm_predictions.parquet")

for df in (brass_pred, glm_weekly):
    df["Departement"] = df["Departement"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(2)

# Aggregate Brass age-level data to weekly departmental totals
brass_weekly = brass_pred.groupby(["Departement", "Year", "Week"], as_index=False).agg(Observed=("Deaths", "sum"), Population=("Population", "sum"), Pred_Brass=("Expected_brass", "sum"))

# Aggregate National GLM predictions to weekly national totals
nat_weekly_total = nat_pred.groupby(["Year", "Week"], as_index=False).agg(Pred_Nat_Direct=("Expected_national", "sum"), Nat_Population=("Population", "sum"))

# Merge Departmental GLM and Brass
comp = glm_weekly.rename(columns={"Joint_observed_weather": "Pred_GLM","Joint_normal_weather": "Pred_GLM_Normal"})[["Departement", "Year", "Week", "Observed", "Pred_GLM"]].merge(
    brass_weekly[["Departement", "Year", "Week", "Population", "Pred_Brass"]], on=["Departement", "Year", "Week"], how="inner")

# Create a naive baseline: distribute national GLM predictions by department population share
comp = comp.merge(nat_weekly_total, on=["Year", "Week"], how="left")
comp["Pred_Nat_Proportional"] = comp["Pred_Nat_Direct"] * (comp["Population"] / comp["Nat_Population"])


# 2. Metric calculation function
def compute_metrics(df, obs_col, pred_col):
    obs = df[obs_col].to_numpy()
    pred = df[pred_col].to_numpy()
    obs_sum, pred_sum = obs.sum(), pred.sum()
    return pd.Series({
        "Observed": obs_sum,
        "Predicted": pred_sum,
        "Bias": pred_sum - obs_sum,
        "Bias_pct": 100 * (pred_sum - obs_sum) / obs_sum if obs_sum > 0 else np.nan,
        "MAE_weekly": mean_absolute_error(obs, pred),
        "RMSE_weekly": np.sqrt(mean_squared_error(obs, pred))
    })


# 3. National Weekly Evaluation (Macro Level)
nat_macro = comp.groupby(["Year", "Week"], as_index=False).agg(Observed=("Observed", "sum"), Pred_GLM_Sum=("Pred_GLM", "sum"), Pred_Brass_Sum=("Pred_Brass", "sum")).merge(nat_weekly_total[["Year", "Week", "Pred_Nat_Direct"]], on=["Year", "Week"], how="left")

nat_rows = []
for year, g in nat_macro.groupby("Year"):
    for label, col in [("National GLM (Direct)", "Pred_Nat_Direct"), ("Dept GLM (Aggregated)", "Pred_GLM_Sum"),("Brass (Aggregated)", "Pred_Brass_Sum")]:
        m = compute_metrics(g, "Observed", col)
        m["Year"] = year
        m["Model"] = label
        nat_rows.append(m)

nat_summary = pd.DataFrame(nat_rows)[["Year", "Model", "Observed", "Predicted", "Bias", "Bias_pct", "MAE_weekly", "RMSE_weekly"]]
print("=== 1. NATIONAL-LEVEL WEEKLY TOTALS ===")
print(nat_summary.round(2).to_string(index=False))


# 4. Departmental Weekly Evaluation (Micro Level: Pooled across all 95 departments)
dep_rows = []
for year, g in comp.groupby("Year"):
    for label, col in [("National GLM (Pop-Proportional)", "Pred_Nat_Proportional"), ("Brass Model", "Pred_Brass"), ("Dept GLM (Local Weather)", "Pred_GLM")]:
        m = compute_metrics(g, "Observed", col)
        m["Year"] = year
        m["Model"] = label
        dep_rows.append(m)

dep_summary = pd.DataFrame(dep_rows)[["Year", "Model", "Observed", "Predicted", "Bias", "Bias_pct", "MAE_weekly", "RMSE_weekly"]]
print("\n=== 2. DEPARTMENTAL WEEKLY TOTALS (POOLED 95 DEPARTMENTS) ===")
print(dep_summary.round(2).to_string(index=False))


# 5. Department-by-Department Head-to-Head
dep_nat_m = comp.groupby("Departement").apply(lambda g: compute_metrics(g, "Observed", "Pred_Nat_Proportional"), include_groups=False).reset_index()
dep_brass_m = comp.groupby("Departement").apply(lambda g: compute_metrics(g, "Observed", "Pred_Brass"), include_groups=False).reset_index()
dep_glm_m = comp.groupby("Departement").apply(lambda g: compute_metrics(g, "Observed", "Pred_GLM"), include_groups=False).reset_index()

h2h = dep_glm_m[["Departement", "Observed", "RMSE_weekly", "MAE_weekly", "Bias_pct"]].rename(
    columns={"RMSE_weekly": "RMSE_GLM", "MAE_weekly": "MAE_GLM", "Bias_pct": "Bias_pct_GLM"}
).merge(dep_brass_m[["Departement", "RMSE_weekly", "MAE_weekly", "Bias_pct"]].rename(columns={"RMSE_weekly": "RMSE_Brass", "MAE_weekly": "MAE_Brass", "Bias_pct": "Bias_pct_Brass"}), on="Departement"
).merge(dep_nat_m[["Departement", "RMSE_weekly", "MAE_weekly", "Bias_pct"]].rename(columns={"RMSE_weekly": "RMSE_Nat", "MAE_weekly": "MAE_Nat", "Bias_pct": "Bias_pct_Nat"}), on="Departement")


In [ ]:
l = []
for i in range(95):
    if h2h[["RMSE_Brass", "RMSE_GLM"]].idxmin(axis=1)[i] == "RMSE_Brass":
        l.append(i+1)
len(l)

In [ ]:
g = population[population["Departement"].isin(l) & population["Week"].isin([1]) & population["Year"].isin([2025])][["Total", "Departement"]].groupby("Departement", as_index=False).agg(Total=("Total", "sum")).astype(int).sort_values("Total")["Departement"].to_list()
h = population[population["Week"].isin([1]) & population["Year"].isin([2025])][["Total", "Departement"]].groupby("Departement", as_index=False).agg(Total=("Total", "sum")).astype(int).sort_values("Total")['Departement'].to_list()
t = []
for x in g:
    t.append(h.index(x))

In [ ]:
t[len(t)//2:]

In [ ]:
mean_squared_error(brass_weekly["Observed"], brass_weekly["Pred_Brass"])